## 10.26反馈：
做的非常好。使用了sklearn库中的OneHotEncoder对词语进行了one-hot编码，举了两个例子。在构建词袋模型时的思路流程是完全正确的，根据分词结果手动构建词汇表并进行one-hot编码。构建词汇表的过程写的很清晰，体现了最基本的原理。但最后计算TF-IDF的代码好像缺失了，运行结果里有计算了TF-IDF矩阵之后的结果，但代码中没有体现。可以进行补充。还有第一段代码的作用可以详细说明一下。

现在你主要使用的是sklearn中的OneHotEncoder函数进行one-hot编码，接下来一周去了解一下sklearn库中的TfidfVectorizer函数，调用这个函数可直接进行编码并生成TF-IDF加权后的词袋矩阵，不需要按照最底层的顺序一步步来做。同时，构建词表也有更简单的命令，将每个句子分词后的结果存放在一个字符串中，词与词之间用空格隔开，这样就可以直接使用.get_feature_names_out()指令创建对应的词汇表。最重要的是多找几个句子，生成每个句子的词袋模型矩阵后进行相似度计算（可以多用几种计算方法），并且尝试将结果可视化，例如生成热力图。
    
除了代码之外，学习一下几种相似度计算的方法（余弦相似度、欧氏距离等），理解计算公式，下周可以给大家讲讲。

One-Hot

In [2]:
from sklearn import preprocessing
import numpy as np
enc=preprocessing.OneHotEncoder()
data = np.array([[0, 0, 3],
                 [1, 1, 0], 
                 [0, 2, 1],
                 [1, 0, 2]])
#先用fit学习数据的特征
enc.fit(data)
#再用transform转换数据
array=enc.transform(data).toarray()
print(array)

[[1. 0. 1. 0. 0. 0. 0. 0. 1.]
 [0. 1. 0. 1. 0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 1. 0. 1. 0. 0.]
 [0. 1. 1. 0. 0. 0. 0. 1. 0.]]


In [ ]:
## 这

In [3]:
from sklearn.preprocessing import OneHotEncoder
# 创建示例数据（纯Python列表）
data = [['红色'], ['蓝色'], ['绿色'], ['红色'], ['蓝色']]
# 创建并训练编码器
#上一段代码中是import全部的sklearn库，这里只import OneHotEncoder
encoder = OneHotEncoder(sparse_output=True)
#上面括号里的是用来控制输出格式的false返回密集矩阵，易于查看操作，而true返回稀疏矩阵，节省内存（默认是true）
#这里讲fit与transform结合在了一起
encoded_data = encoder.fit_transform(data)
# 查看结果
print("原始数据:", data)
print("编码结果:")
print(encoded_data)
print("特征名称:", encoder.get_feature_names_out(['颜色']))
# get_feature_names_out(['颜色']) 中的 ['颜色'] 是输入特征的名称
# 它告诉编码器："这些编码后的特征来自于名为'颜色'的原始特征"

原始数据: [['红色'], ['蓝色'], ['绿色'], ['红色'], ['蓝色']]
编码结果:
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5 stored elements and shape (5, 3)>
  Coords	Values
  (0, 0)	1.0
  (1, 2)	1.0
  (2, 1)	1.0
  (3, 0)	1.0
  (4, 2)	1.0
特征名称: ['颜色_红色' '颜色_绿色' '颜色_蓝色']


In [4]:
import numpy as np
# 创建学生数据
student_data = np.array([
    ['大一', 'A'],
    ['大二', 'B'], 
    ['大三', 'A'],
    ['大一', 'C'],
    ['大二', 'B']
])
print("原始数据:")
print(student_data)
# 创建并训练编码器
encoder = OneHotEncoder(sparse_output=False)
encoded_data = encoder.fit_transform(student_data)
print("\nOne-Hot编码结果:")
print(encoded_data)
print("\n特征含义:")
print(encoder.get_feature_names_out(['年级', '成绩等级']))
# 测试新数据
new_student = np.array([['大一', 'B']])
new_encoded = encoder.transform(new_student)
print(f"\n新数据 ['大一', 'B'] 编码结果:")
print(new_encoded[0])

原始数据:
[['大一' 'A']
 ['大二' 'B']
 ['大三' 'A']
 ['大一' 'C']
 ['大二' 'B']]

One-Hot编码结果:
[[1. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 1. 0.]
 [0. 1. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 1.]
 [0. 0. 1. 0. 1. 0.]]

特征含义:
['年级_大一' '年级_大三' '年级_大二' '成绩等级_A' '成绩等级_B' '成绩等级_C']

新数据 ['大一', 'B'] 编码结果:
[1. 0. 0. 0. 1. 0.]


Bag of words

In [29]:
from collections import defaultdict  # 使用默认字典来统计词频
import jieba
# 创建词袋模型的函数
def simple_bag_of_words(documents):
    """
    简单的词袋模型实现
    输入：文档列表，例如 ["hello world", "hello python"]
    输出：词袋向量和词汇表
    """
    #上方是文档字符串，用于描述函数的输入输出
    # 步骤1：对中文文档进行分词
    print("=== 分词结果 ===")
    segmented_docs = []  # 存储分词后的文档
    for doc in documents:
        words = jieba.lcut(doc)  # 使用jieba进行中文分词
        segmented_docs.append(words)
        print(f"文档 '{doc}' 分词结果: {words}")
    print()  # 空行
    # 步骤2：构建词汇表
    vocabulary = {}  # 创建一个空字典来存储词汇表
    index = 0        # 词汇索引从0开始
    # 遍历所有分词后的文档
    for words in segmented_docs:    # 对每个文档的分词结果
        for word in words:          # 对每个单词
            if word not in vocabulary:  # 如果单词不在词汇表中
                vocabulary[word] = index  # 将单词添加到词汇表
                index += 1           # 索引加1
    print("词汇表:", vocabulary)  # 打印词汇表
    # 步骤3：创建词袋向量
    bag_of_words_vectors = [] # 创建空列表来存储所有文档的向量
    for i,words in enumerate(segmented_docs):  # 对每个文档
        # 初始化一个全0向量，长度等于词汇表大小
        vector = [0] * len(vocabulary)
        # 统计当前文档中每个单词的出现次数
        for word in words:  # 遍历文档中的每个单词
            if word in vocabulary:  # 如果单词在词汇表中
                # 找到单词在词汇表中的位置，并在对应位置计数加1
                vector[vocabulary[word]] += 1
        bag_of_words_vectors.append(vector)  # 将当前文档的向量添加到结果列表
        print(f"文档 '{doc}' 的词袋向量: {vector}")
    return bag_of_words_vectors, vocabulary,segmented_docs
# 使用示例
if __name__ == "__main__":
    # 示例文档集
    documents = [
        "我喜欢编程",
        "编程很有趣", 
        "我喜欢学习"
    ]
    print("输入文档:", documents)
    print("\n处理过程:")
    #调用词袋模型函数
    vectors, vocab, segmented_docs = simple_bag_of_words(documents)
    print("\n=== 最终结果 ===")
    print("词汇表:", vocab)
    print("所有文档的词袋向量:")
    for i, vector in enumerate(vectors):
        print(f"文档{i+1}: {vector}")
      # 使用已经分好词的结果计算TF-IDF
    tfidf_matrix, vectorizer = calculate_tfidf_from_segmented(segmented_docs, vocab, documents)

输入文档: ['我喜欢编程', '编程很有趣', '我喜欢学习']

处理过程:
=== 分词结果 ===
文档 '我喜欢编程' 分词结果: ['我', '喜欢', '编程']
文档 '编程很有趣' 分词结果: ['编程', '很', '有趣']
文档 '我喜欢学习' 分词结果: ['我', '喜欢', '学习']

词汇表: {'我': 0, '喜欢': 1, '编程': 2, '很': 3, '有趣': 4, '学习': 5}
文档 '我喜欢学习' 的词袋向量: [1, 1, 1, 0, 0, 0]
文档 '我喜欢学习' 的词袋向量: [0, 0, 1, 1, 1, 0]
文档 '我喜欢学习' 的词袋向量: [1, 1, 0, 0, 0, 1]

=== 最终结果 ===
词汇表: {'我': 0, '喜欢': 1, '编程': 2, '很': 3, '有趣': 4, '学习': 5}
所有文档的词袋向量:
文档1: [1, 1, 1, 0, 0, 0]
文档2: [0, 0, 1, 1, 1, 0]
文档3: [1, 1, 0, 0, 0, 1]

=== 使用已分词结果计算TF-IDF ===
分词后的文本:
文档1: 我 喜欢 编程
文档2: 编程 很 有趣
文档3: 我 喜欢 学习

TF-IDF矩阵形状: (3, 6)
特征词汇: ['我' '喜欢' '编程' '很' '有趣' '学习']

TF-IDF矩阵:
       我     喜欢     编程    很     有趣     学习
文档1  0.0  0.707  0.707  0.0  0.000  0.000
文档2  0.0  0.000  0.605  0.0  0.796  0.000
文档3  0.0  0.605  0.000  0.0  0.000  0.796
